# 09 — Comparaison de stratégies lag 1

Ce notebook compare plusieurs règles de portefeuille avec des prédictions walk-forward hors échantillon. Le lag est strict : la prédiction de J-1 est appliquée au gap de J. Les stratégies sont comparées à coûts de 5, 10 et 20 bp. Le test final ne sert pas à choisir les seuils : les paramètres doivent être fixés avant l'interprétation.

In [15]:
import os, json
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

PROJECT = r"C:\Users\semy4\OneDrive\Bureau\Fintech_project"
DATA = os.path.join(PROJECT, "data", "processed")
df = pd.read_csv(os.path.join(DATA, "DATASET_MODELISATION_2020_2022.csv"))
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)
with open(os.path.join(DATA, "config_modelisation.json"), encoding="utf-8") as f:
    cfg = json.load(f)
features = [c for c in cfg["features"] if c in df.columns]

_absentes = [c for c in cfg["features"] if c not in df.columns]
if _absentes:
    raise KeyError(f"{len(_absentes)} features déclarées dans config_modelisation.json "
                   f"sont absentes du CSV : {_absentes}\nRéexécuter le notebook 05.")
purge = cfg["purge_jours"]

def logit():
    return Pipeline([("imp", SimpleImputer(strategy="median")), ("std", StandardScaler()), ("clf", LogisticRegression(max_iter=2000))])

def folds(dates, n=8, test_size=60, min_train=250):
    days = np.sort(pd.unique(dates)); result = []; end = len(days)
    for _ in range(n):
        split = end - test_size
        if split - purge < min_train: break
        result.append((days[:split-purge], days[split:end])); end = split
    return result[::-1]

# --------------------------------------------------------------------------
# Le walk-forward NE DOIT PAS toucher le bloc de test, sinon l'évaluation
# finale du notebook 06 §9 n'est plus en aveugle. On restreint donc toutes
# les prédictions hors échantillon aux blocs train / purge_1 / valid.
# --------------------------------------------------------------------------
BLOCS_WF = ["train", "purge_1", "valid"]
d = df[df["bloc"].isin(BLOCS_WF)].dropna(subset=["y_gap", "gap"]).copy()
blocks = []
_plis = folds(d["Date"])
_dernier = pd.Timestamp(_plis[-1][1][-1])
_premier = df.loc[df["bloc"] == "test", "Date"].min()
print(f"Dernier jour du walk-forward : {_dernier.date()}")
print(f"Premier jour du bloc test    : {_premier.date()}")
assert _dernier < _premier, \
    "Le walk-forward empiète sur le bloc de test : le §9 ne serait plus en aveugle."
print("[OK] Le bloc de test est intact.")

for train_days, test_days in _plis:
    train = d[d["Date"].isin(train_days)]
    test = d[d["Date"].isin(test_days)].copy()
    if len(train) < 250 or test["y_gap"].nunique() < 2:
        continue
    model = clone(logit()).fit(train[features], train["y_gap"])
    test["proba"] = model.predict_proba(test[features])[:, 1]
    blocks.append(test)
p = pd.concat(blocks).sort_values(["Ticker", "Date"]).reset_index(drop=True)
p["proba_oos"] = p["proba"]
p_full = p.copy()
p["proba_lag1"] = p.groupby("Ticker")["proba_oos"].shift(1)
p = p.dropna(subset=["proba_lag1"]).copy()
p["proba"] = p["proba_lag1"]
print(f"Prédictions lag 1 : {len(p):,} lignes, {p['Date'].nunique()} jours")
print(f"AUC descriptive du signal décalé : {roc_auc_score(p['y_gap'], p['proba']):.4f}")

Dernier jour du walk-forward : 2021-11-30
Premier jour du bloc test    : 2021-12-06
[OK] Le bloc de test est intact.
Prédictions lag 1 : 895 lignes, 179 jours
AUC descriptive du signal décalé : 0.5119


## Stratégies testées

Les seuils sont fixés à l'avance : 0,55/0,45 et 0,60/0,40. La stratégie long-only ne prend jamais de position short. Les poids sont normalisés par jour afin que le portefeuille ait une exposition comparable entre les journées.

In [16]:
def make_positions(data, name):
    x = data.copy()
    e = x["proba"] - 0.5
    if name == "long_only_055":
        raw = (x["proba"] > 0.55).astype(float)
    elif name == "extreme_ls_060":
        raw = np.where(x["proba"] > 0.60, 1.0, np.where(x["proba"] < 0.40, -1.0, 0.0))
    elif name == "proportional_ls":
        raw = np.clip(e / 0.15, -1.0, 1.0)
    elif name == "risk_scaled_ls":
        vol = x["vol_20"].replace(0, np.nan).fillna(x["vol_20"].median())
        raw = np.clip(e / 0.15, -1.0, 1.0) * (0.02 / vol).clip(0.25, 3.0)
    elif name == "long_only_extreme":
        raw = (x["proba"] > 0.60).astype(float)
    else:
        raise ValueError(name)
    x["raw_position"] = raw
    # transform() au lieu de apply() : plus rapide, et sans FutureWarning sur
    # l'inclusion des colonnes de groupement.
    gross = x.groupby("Date")["raw_position"].transform(lambda s: s.abs().sum())
    x["position"] = np.where(gross > 0, x["raw_position"] / gross, 0.0)
    return x

strategies = ["long_only_055", "long_only_extreme", "extreme_ls_060", "proportional_ls", "risk_scaled_ls"]

def evaluate(data, cost_bp):
    x = data.sort_values(["Ticker", "Date"]).copy()
    x["turnover"] = (x.groupby("Ticker")["position"].diff().abs().fillna(x["position"].abs()))
    x["gross_pnl"] = x["position"] * x["gap"]
    x["cost"] = x["turnover"] * cost_bp / 10000
    x["net_pnl"] = x["gross_pnl"] - x["cost"]
    daily = x.groupby("Date").agg(net=("net_pnl", "sum"), gross=("gross_pnl", "sum"), cost=("cost", "sum"), turnover=("turnover", "sum"), n_pos=("position", lambda s: (s != 0).sum()))
    r = daily["net"]
    equity = (1 + r).cumprod()
    sharpe = r.mean() / r.std() * np.sqrt(252) if r.std() > 0 else np.nan
    drawdown = (equity / equity.cummax() - 1).min()
    from scipy.stats import ttest_1samp
    p_value = ttest_1samp(r, 0, nan_policy="omit").pvalue if len(r) > 1 else np.nan
    return {"cost_bp": cost_bp, "annual_return": r.mean() * 252, "sharpe": sharpe, "max_drawdown": drawdown, "win_rate": (r > 0).mean(), "p_value_mean": p_value, "mean_daily_pnl": r.mean(), "total_turnover": daily["turnover"].sum(), "days": len(r), "positions": int(daily["n_pos"].sum())}


In [17]:
rows = []
for strategy in strategies:
    positions = make_positions(p, strategy)
    for cost in [5, 10, 20]:
        row = evaluate(positions, cost)
        row["strategy"] = strategy
        rows.append(row)
results = pd.DataFrame(rows)[["strategy", "cost_bp", "annual_return", "sharpe", "max_drawdown", "win_rate", "p_value_mean", "mean_daily_pnl", "total_turnover", "days", "positions"]]
display(results.sort_values(["cost_bp", "sharpe"], ascending=[True, False]).round(4))
out = os.path.join(DATA, "BACKTEST_STRATEGIES_LAG1_RESULTS.csv")
results.to_csv(out, index=False)
print("Export :", out)

,strategy,cost_bp,annual_return,sharpe,max_drawdown,win_rate,p_value_mean,mean_daily_pnl,total_turnover,days,positions
3,long_only_extreme,5,0.1533,1.0145,-0.1086,0.5307,0.3937,0.0006,204.0333,179,484
0,long_only_055,5,0.1295,0.8967,-0.0818,0.5642,0.4508,0.0005,158.0667,179,618
6,extreme_ls_060,5,0.0885,0.5818,-0.1510,0.5587,0.6245,0.0004,229.6333,179,552
9,proportional_ls,5,0.0410,0.3163,-0.1263,0.5642,0.7901,0.0002,173.3295,179,895
12,risk_scaled_ls,5,0.0204,0.1732,-0.1252,0.5866,0.8841,0.0001,175.1424,179,895
1,long_only_055,10,0.0182,0.1260,-0.1182,0.5419,0.9155,0.0001,158.0667,179,618
4,long_only_extreme,10,0.0097,0.0639,-0.1664,0.5196,0.9571,0.0000,204.0333,179,484
7,extreme_ls_060,10,-0.0732,-0.4810,-0.2112,0.5140,0.6857,-0.0003,229.6333,179,552
10,proportional_ls,10,-0.0810,-0.6234,-0.1740,0.5307,0.6000,-0.0003,173.3295,179,895
13,risk_scaled_ls,10,-0.1029,-0.8708,-0.1695,0.5363,0.4640,-0.0004,175.1424,179,895


Export : C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\processed\BACKTEST_STRATEGIES_LAG1_RESULTS.csv


## Lecture

La meilleure stratégie ne doit pas être choisie uniquement selon le Sharpe du test. Il faut privilégier une règle fixée avant le test, stable à 5/10/20 bp, avec un drawdown acceptable et un résultat cohérent par sous-période. Un Sharpe positif à 0 bp mais négatif à 10 bp signifie que le signal est trop faible pour être exploité après coûts.

## Analyse des retards de 1 à 5 jours

Chaque retard est calculé à partir de la prédiction walk-forward originale : `proba de J-k` est appliquée au gap de J. Les mêmes stratégies et coûts sont utilisés pour comparer uniquement l'effet du délai.

In [18]:
lag_rows = []
for lag in range(1, 6):
    lag_data = p_full.copy()
    lag_data["proba"] = lag_data.groupby("Ticker")["proba_oos"].shift(lag)
    lag_data = lag_data.dropna(subset=["proba"]).copy()
    for strategy in strategies:
        positions = make_positions(lag_data, strategy)
        for cost in [5, 10, 20]:
            row = evaluate(positions, cost)
            row.update({"lag": lag, "strategy": strategy})
            lag_rows.append(row)
lag_results = pd.DataFrame(lag_rows)
lag_results = lag_results[["lag", "strategy", "cost_bp", "annual_return", "sharpe", "max_drawdown", "win_rate", "p_value_mean", "mean_daily_pnl", "total_turnover", "days", "positions"]]
display(lag_results.sort_values(["cost_bp", "sharpe"], ascending=[True, False]).round(4))
lag_out = os.path.join(DATA, "BACKTEST_STRATEGIES_LAG1_TO_LAG5_RESULTS.csv")
lag_results.to_csv(lag_out, index=False)
print("Export :", lag_out)

,lag,strategy,cost_bp,annual_return,sharpe,max_drawdown,win_rate,p_value_mean,mean_daily_pnl,total_turnover,days,positions
45,4,long_only_055,5,0.3591,2.1545,-0.0753,0.5739,0.0735,0.0014,155.2667,176,605
15,2,long_only_055,5,0.2888,1.9141,-0.0697,0.6067,0.1095,0.0011,157.2667,178,613
54,4,proportional_ls,5,0.2438,1.8786,-0.0943,0.5966,0.1182,0.0010,170.7286,176,880
30,3,long_only_055,5,0.2262,1.7366,-0.0551,0.5706,0.1473,0.0009,156.4667,177,610
48,4,long_only_extreme,5,0.2545,1.4938,-0.1040,0.5170,0.2136,0.0010,201.7333,176,472
...,...,...,...,...,...,...,...,...,...,...,...,...
71,5,proportional_ls,20,-0.3702,-2.9468,-0.2786,0.4057,0.0150,-0.0015,169.6315,175,875
29,2,risk_scaled_ls,20,-0.3516,-3.0587,-0.2540,0.4270,0.0110,-0.0014,174.6658,178,890
74,5,risk_scaled_ls,20,-0.4033,-3.5482,-0.2822,0.3943,0.0035,-0.0016,172.1370,175,875
68,5,extreme_ls_060,20,-0.5314,-3.9512,-0.3503,0.3657,0.0012,-0.0021,224.3333,175,539


Export : C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\processed\BACKTEST_STRATEGIES_LAG1_TO_LAG5_RESULTS.csv


## Benchmarks, robustesse et significativité

Cette section complète le cahier des charges : benchmark Buy & Hold sur les gaps disponibles, analyse par période et par ticker, et test t de la moyenne des rendements journaliers. Un indice de marché externe doit être ajouté séparément si une série SPY/S&P 500 synchronisée est disponible.

In [19]:
# Benchmark Buy & Hold égal-pondéré sur l'univers observé.
# Il s'agit d'un benchmark sur les gaps (pas d'un rendement close-to-close complet).
benchmark = p.copy()
benchmark["position"] = 1.0 / benchmark.groupby("Date")["Ticker"].transform("nunique")
benchmark_rows = []
for cost in [0, 5, 10, 20]:
    row = evaluate(benchmark, cost)
    row["benchmark"] = "buy_hold_equal_weight_gap"
    benchmark_rows.append(row)
benchmark_results = pd.DataFrame(benchmark_rows)
display(benchmark_results.round(4))
benchmark_out = os.path.join(DATA, "BACKTEST_BENCHMARKS_RESULTS.csv")
benchmark_results.to_csv(benchmark_out, index=False)
print("Export :", benchmark_out)
print("Indice de marché externe : non disponible dans DATASET_MODELISATION_2020_2022.csv")

,cost_bp,annual_return,sharpe,max_drawdown,win_rate,p_value_mean,mean_daily_pnl,total_turnover,days,positions,benchmark
0,0,0.2713,2.0721,-0.0551,0.6145,0.0825,0.0011,1.0,179,895,buy_hold_equal_weight_gap
1,5,0.2706,2.0652,-0.0551,0.6145,0.0835,0.0011,1.0,179,895,buy_hold_equal_weight_gap
2,10,0.2698,2.0583,-0.0551,0.6145,0.0845,0.0011,1.0,179,895,buy_hold_equal_weight_gap
3,20,0.2684,2.0444,-0.0551,0.6145,0.0866,0.0011,1.0,179,895,buy_hold_equal_weight_gap


Export : C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\processed\BACKTEST_BENCHMARKS_RESULTS.csv
Indice de marché externe : non disponible dans DATASET_MODELISATION_2020_2022.csv


In [20]:
# Robustesse avec une règle fixée à l'avance : lag 1, long_only_055, coût 10 bp.
fixed_strategy = make_positions(p, "long_only_055")
robust_rows = []
for year, group in fixed_strategy.groupby(fixed_strategy["Date"].dt.year):
    if group["Date"].nunique() > 1:
        row = evaluate(group, 10)
        row.update({"dimension": "year", "group": str(year)})
        robust_rows.append(row)
for ticker, group in fixed_strategy.groupby("Ticker"):
    if group["Date"].nunique() > 1:
        row = evaluate(group, 10)
        row.update({"dimension": "ticker", "group": ticker})
        robust_rows.append(row)
robustness_results = pd.DataFrame(robust_rows)
display(robustness_results[["dimension", "group", "annual_return", "sharpe", "max_drawdown", "win_rate", "p_value_mean", "days", "positions"]].round(4))
robust_out = os.path.join(DATA, "BACKTEST_ROBUSTNESS_YEAR_TICKER_RESULTS.csv")
robustness_results.to_csv(robust_out, index=False)
print("Export :", robust_out)

,dimension,group,annual_return,sharpe,max_drawdown,win_rate,p_value_mean,days,positions
0,year,2021,0.0182,0.1260,-0.1182,0.5419,0.9155,179,618
1,ticker,AAPL,-0.0391,-1.4630,-0.0384,0.3408,0.2192,179,119
2,ticker,AMZN,-0.0431,-1.3737,-0.0435,0.3408,0.2485,179,116
3,ticker,META,-0.0378,-0.6923,-0.0636,0.3743,0.5603,179,123
4,ticker,NVDA,0.1364,2.1962,-0.0246,0.4358,0.0658,179,130
5,ticker,TSLA,0.0017,0.0283,-0.0530,0.4134,0.9810,179,130


Export : C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\processed\BACKTEST_ROBUSTNESS_YEAR_TICKER_RESULTS.csv


## Benchmark de marché : SPY / S&P 500

SPY est un ETF qui réplique le S&P 500. Pour rester comparable avec le projet, le benchmark utilise le rendement overnight : `Open_t / Close_{t-1} - 1`. Les données sont téléchargées depuis Yahoo Finance et exportées séparément.

In [21]:
import yfinance as yf

spy = yf.download(
    "SPY",
    start="2020-01-01",
    end="2023-01-01",
    auto_adjust=False,
    progress=False,
    threads=False,
)
if spy.empty:
    raise RuntimeError("Yahoo Finance n'a retourné aucune donnée pour SPY.")
if isinstance(spy.columns, pd.MultiIndex):
    spy.columns = spy.columns.get_level_values(0)
spy = spy.reset_index()
spy["Date"] = pd.to_datetime(spy["Date"], utc=True).dt.tz_localize(None)
spy["gap"] = spy["Open"] / spy["Close"].shift(1) - 1
spy_benchmark = spy[["Date", "gap"]].dropna().copy()
spy_benchmark["Ticker"] = "SPY"
spy_benchmark["position"] = 1.0
spy_benchmark = spy_benchmark[spy_benchmark["Date"].isin(p["Date"])].copy()
if spy_benchmark.empty:
    raise RuntimeError("Aucune date SPY ne correspond aux dates du backtest.")

spy_rows = []
for cost in [0, 5, 10, 20]:
    row = evaluate(spy_benchmark, cost)
    row["benchmark"] = "SPY_overnight_gap"
    spy_rows.append(row)
spy_results = pd.DataFrame(spy_rows)
display(spy_results.round(4))
spy_out = os.path.join(DATA, "BACKTEST_SPY_BENCHMARK_RESULTS.csv")
spy_results.to_csv(spy_out, index=False)
print("Export SPY :", spy_out)

strategy_lag1 = make_positions(p, "long_only_055")
comparison_rows = []
for cost in [0, 5, 10, 20]:
    row = evaluate(strategy_lag1, cost)
    row["benchmark"] = "sentiment_lag1_long_only_055"
    comparison_rows.append(row)
comparison = pd.concat([spy_results, pd.DataFrame(comparison_rows)], ignore_index=True)
display(comparison[["benchmark", "cost_bp", "annual_return", "sharpe", "max_drawdown", "win_rate", "p_value_mean"]].round(4))
comparison_out = os.path.join(DATA, "BACKTEST_SPY_VS_SENTIMENT_RESULTS.csv")
comparison.to_csv(comparison_out, index=False)
print("Export comparaison :", comparison_out)

,cost_bp,annual_return,sharpe,max_drawdown,win_rate,p_value_mean,mean_daily_pnl,total_turnover,days,positions,benchmark
0,0,0.0861,1.2520,-0.0236,0.6034,0.2928,0.0003,1.0,179,179,SPY_overnight_gap
1,5,0.0854,1.2403,-0.0236,0.6034,0.2973,0.0003,1.0,179,179,SPY_overnight_gap
2,10,0.0847,1.2286,-0.0236,0.6034,0.3018,0.0003,1.0,179,179,SPY_overnight_gap
3,20,0.0833,1.2050,-0.0236,0.6034,0.3112,0.0003,1.0,179,179,SPY_overnight_gap


Export SPY : C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\processed\BACKTEST_SPY_BENCHMARK_RESULTS.csv


,benchmark,cost_bp,annual_return,sharpe,max_drawdown,win_rate,p_value_mean
0,SPY_overnight_gap,0,0.0861,1.2520,-0.0236,0.6034,0.2928
1,SPY_overnight_gap,5,0.0854,1.2403,-0.0236,0.6034,0.2973
2,SPY_overnight_gap,10,0.0847,1.2286,-0.0236,0.6034,0.3018
3,SPY_overnight_gap,20,0.0833,1.2050,-0.0236,0.6034,0.3112
4,sentiment_lag1_long_only_055,0,0.2407,1.6657,-0.0663,0.5922,0.1621
5,sentiment_lag1_long_only_055,5,0.1295,0.8967,-0.0818,0.5642,0.4508
6,sentiment_lag1_long_only_055,10,0.0182,0.1260,-0.1182,0.5419,0.9155
7,sentiment_lag1_long_only_055,20,-0.2043,-1.4131,-0.2033,0.5028,0.2353


Export comparaison : C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\processed\BACKTEST_SPY_VS_SENTIMENT_RESULTS.csv
